**Step 1: This is used to prepare the dataset by combining and mapping patient, episode and contact using the tables `sak`, `pasient`, `preparat`, `forordning`, `resept`, `journal`, and `diagnose`**
- We link the patient, episode, contact.
- Calculate date of birth at episode level 
- Clean atc codes

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
load_dotenv()

data_dir = os.getenv("DATA_DIR")
MAIN_DATASET_PATH = os.getenv("MAIN_DATASET_PATH")

pasient_cols = [
    "nr",
    "fdt",
    "morfdt",
    "farfdt",
]
pasient_dtype = {
    "nr": "float64",
    "fdt": "object",
    "morfdt": "object",
    "farfdt": "object",
}

sak_cols = [
    "pasient",
    "nr",
    "kjonn",
    "henvdato",
    "igangdato",
    "avsldato",
]
sak_dtype = {
    "pasient": "int64",
    "nr": "float64",
    "kjonn": "float64",
    "henvdato": "object",
    "igangdato": "object",
    "avsldato": "object",
}

forordning_cols = ["pasientnr", "preparatid", "saknr"]
forordning_dtype = {
    "pasientnr": "float64",
    "preparatid": "object",
    "saknr": "float64",
}

resept_cols = ["forordningnr", "preparatid"]
resept_dtype = {
    "forordningnr": "object",
    "preparatid": "object",
}

preparat_cols = ["id", "atckode", "atcnavn"]
preparat_dtype = {
    "id": "object",
    "atckode": "object",
    "atcnavn": "object",
}

diagnose_cols = ["pasient", "sak", "opphold", "diagnose", "akse", "dato", "hoved"]
diagnose_dtype = {
    "pasient": "float64",
    "sak": "float64",
    "opphold": "float64",
    "diagnose": "object",
    "akse": "object",
    "dato": "object",
    "hoved": "object",
}

opphold_cols = ["nr", "pasient", "sak", "igangdato", "avsldato"]
opphold_dtype = {
    "nr": "float64",
    "pasient": "float64",
    "sak": "float64",
    "igangdato": "object",
    "avsldato": "object",
}


def main():
    # Read CSVs
    pasient = dd.read_csv(
        os.path.join(data_dir, "pasient.csv"),
        dtype=pasient_dtype,
        usecols=pasient_cols,
    ).rename(columns={"nr": "pasient_nr"})

    sak = dd.read_csv(
        os.path.join(data_dir, "sak.csv"),
        dtype=sak_dtype,
        usecols=sak_cols,
    ).rename(columns={col: f"sak_{col}" for col in sak_cols})

    forordning = dd.read_csv(
        os.path.join(data_dir, "forordning.csv"),
        dtype=forordning_dtype,
        usecols=forordning_cols,
    )

    resept = dd.read_csv(
        os.path.join(data_dir, "resept.csv"),
        dtype=resept_dtype,
        usecols=resept_cols,
    )

    preparat = (
        dd.read_csv(
            os.path.join(data_dir, "preparat.csv"),
            dtype=preparat_dtype,
            usecols=preparat_cols,
        )
        .assign(
            atckode=lambda df: (
                df["atckode"]
                .str.replace(" ", "", regex=False)
                .str.replace("O", "0")
                .str.upper()
                .str.strip()
            )
        )
        .rename(columns={"id": "preparatid"})
    )

    diagnose = (
        dd.read_csv(
            os.path.join(data_dir, "diagnose.csv"),
            dtype=diagnose_dtype,
            usecols=diagnose_cols,
        )
        .rename(columns={col: f"diag_{col}" for col in diagnose_cols})
        .assign(diag_dato=lambda df: dd.to_datetime(df["diag_dato"], errors="coerce"))
    )

    opphold = (
        dd.read_csv(
            os.path.join(data_dir, "opphold.csv"),
            dtype=opphold_dtype,
            usecols=opphold_cols,
        )
        .rename(
            columns={
                "nr": "opphold_id",
                "pasient": "opphold_pasient",
                "sak": "opphold_sak",
                "igangdato": "opphold_start_date",
                "avsldato": "opphold_end_date",
            }
        )
        .assign(
            opphold_start_date=lambda df: dd.to_datetime(
                df["opphold_start_date"], errors="coerce"
            ),
            opphold_end_date=lambda df: dd.to_datetime(
                df["opphold_end_date"], errors="coerce"
            ),
        )
    )

    # Build pasient_sak
    pasient = pasient.assign(
        fdt=lambda df: dd.to_datetime(df["fdt"], errors="coerce"),
        morfdt=lambda df: dd.to_datetime(df["morfdt"], errors="coerce"),
        farfdt=lambda df: dd.to_datetime(df["farfdt"], errors="coerce"),
    )

    sak = sak.assign(
        sak_henvdato=lambda df: dd.to_datetime(df["sak_henvdato"], errors="coerce"),
        sak_igangdato=lambda df: dd.to_datetime(df["sak_igangdato"], errors="coerce"),
        sak_avsldato=lambda df: dd.to_datetime(df["sak_avsldato"], errors="coerce"),
    )

    pasient_sak = (
        dd.merge(pasient, sak, left_on="pasient_nr", right_on="sak_pasient", how="left")
        .assign(
            start_date=lambda df: df["sak_henvdato"].fillna(df["sak_igangdato"]),
            patient_age=lambda df: (df["start_date"] - df["fdt"]).dt.days / 365.25,
            mor_age=lambda df: (df["start_date"] - df["morfdt"]).dt.days / 365.25,
            far_age=lambda df: (df["start_date"] - df["farfdt"]).dt.days / 365.25,
        )
        .drop(columns=["morfdt", "farfdt", "sak_henvdato", "start_date"])
    )

    # Combine forordning → resept → preparat
    forord_prep = dd.merge(forordning, resept, on="preparatid", how="left")
    forord_prep = dd.merge(forord_prep, preparat, on="preparatid", how="left")

    pasient_sak_full = dd.merge(
        pasient_sak, forord_prep, left_on="pasient_nr", right_on="pasientnr", how="left"
    )

    # Merge diagnose + opphold first
    diag_with_opphold = dd.merge(
        diagnose,
        opphold,
        left_on=["diag_opphold", "diag_sak", "diag_pasient"],
        right_on=["opphold_id", "opphold_sak", "opphold_pasient"],
        how="left",
    )

    # Merge into pasient_sak_full and compute diagnosis age
    full_data = (
        dd.merge(
            pasient_sak_full,
            diag_with_opphold,
            left_on=["pasient_nr", "sak_nr"],
            right_on=["diag_pasient", "diag_sak"],
            how="left",
        )
        .assign(
            patient_diagnose_age=lambda df: (df["diag_dato"] - df["fdt"]).dt.days
            / 365.25
        )
        .drop(columns=["fdt"])
    )

    # Select final columns and write
    final_columns = [
        "pasient_nr",
        "sak_nr",
        "sak_kjonn",
        "patient_age",
        "sak_igangdato",
        "sak_avsldato",
        "atckode",
        "atcnavn",
        "diag_diagnose",
        "diag_akse",
        "diag_hoved",
        "patient_diagnose_age",
        "diag_dato",
        "opphold_id",
        "opphold_start_date",
        "opphold_end_date",
    ]
    final_dd = full_data[final_columns]

    output_file = os.path.join(MAIN_DATASET_PATH + "/dato_opphold_forordning.parquet")
    final_dd = final_dd.repartition(npartitions=1)
    final_dd.to_parquet(output_file, engine="pyarrow", write_index=False)
    print(f"Data successfully saved to {output_file}")


if __name__ == "__main__":
    main()

**Step 2: To count the patients and contacts with dates ≥ 1995**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *

SRC_PARQUET = os.getenv("MAIN_DATASET")
DATA_DIR = os.getenv("DATA_DIR")
df = dd.read_parquet(SRC_PARQUET, engine="pyarrow")
print(f"patient: {df["pasient_nr"].nunique().compute()}, contacts: {df["opphold_id"].nunique().compute()}")
df["sak_igangdato"] = dd.to_datetime(df["sak_igangdato"], errors="coerce")
df["sak_avsldato"] = dd.to_datetime(df["sak_avsldato"], errors="coerce")
cutoff = "1995-01-01"
df = df[(df["sak_igangdato"] >= cutoff)]
print(f"Unique patient ≥ 1995: {df["pasient_nr"].nunique().compute()}, unique contacts: {df["opphold_id"].nunique().compute()}")